# Traitement des données

Dans ce notebook, nous mettons en place plusieurs étapes de préparation et d’analyse des données avant la modélisation :

1. **Chargement des données :**  
   - Importer le fichier CSV contenant l’ensemble des variables (climatiques, économiques, etc.).
   - Convertir la colonne Date au format datetime et trier les observations.

2. **Analyse des valeurs manquantes :**  
   - Identifier la proportion de NaN par colonne.
   - Décider de la stratégie de traitement (suppression ou imputation si necessaire).

3. **Visualisation des distributions :**  
   - Examiner la répartition des variables clés (histogrammes, boxplots) pour détecter d’éventuelles anomalies ou asymétries importantes.

4. **Normalisation des variables numériques :**  
   - Appliquer la méthode StandardScaler sur les variables pertinentes pour les ramener à une moyenne de 0 et un écart type de 1.

5. **Sélection de la période d’étude (Panel) :**  
   - Ne conserver que la fenêtre temporelle souhaitée (2003-01 à 2023-12).
   - Vérifier la cohérence des données pour chaque mine (13 mines au total).

6. **Statistiques descriptives :**  
   - Calculer les mesures de tendance centrale et de dispersion (moyenne, médiane, écart type, etc.).
   - Fournir un aperçu global du comportement des variables.

7. **Corrélation :**  
   - Étudier les relations linéaires entre variables clés (matrice de corrélation, heatmap).
   - Identifier d’éventuelles multicolinéarités et variables redondantes.

Ce pipeline garantit une **qualité** et une **cohérence** optimales pour nos analyses et nos futurs modèles prédictifs.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

## 1. Chargement des données

In [ ]:
# Adaptez le chemin vers votre fichier CSV
df = pd.read_csv(r"./data/final_merged_data1.csv", sep=";")

print("Aperçu des données :")
print(df.head())

# Convertir la colonne Date en datetime
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m')  # Adapter le format si nécessaire
df = df.sort_values('Date')

# Afficher la liste des colonnes disponibles
print("\nListe des colonnes :")
print(df.columns.tolist())


In [ ]:
df.rename(columns={
    "Refined Copper Values (1)": "refined_copp",
    "Blister Copper Values (2)": "blister_copp",
    "Bulk Copper Values (3)": "bulk_copp",
    "Totals": "export_tot"
}, inplace=True)

# Vérifions la liste des colonnes pour adapter les variables
print("\nColonnes disponibles :")
print(df.columns.tolist())

## 2. Analyse des valeurs manquantes

In [ ]:
print("\nSomme des valeurs manquantes par colonne :")
print(df.isnull().sum())


## 3. Visualisation des distributions

In [ ]:
# Sélection de quelques variables clés pour la modélisation
key_vars = ['Production', 'Temperature', 'Precipitation', 'Cloud_Cover', 'Diurnal_Temp_Range', 
            'Frost_Days', 'Potential_Evapotranspiration', 'Anomalie_Cloud_Cover',
            'Anomalie_Diurnal_Temp_Range',  'Anomalie_Frost_Days', 
            'Anomalie_Potential_Evapotranspiration', 'Anomalie_Precipitation', 'Anomalie_Temperature',
            'Temp_Min', 'Temp_Max', 'Vapor_Pressure', 'Wet_Days', 'Anomalie_Vapor_Pressure',
            'Anomalie_Wet_Days', 'prix_lme', 'export_tot'
           ]

for var in key_vars:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[var].dropna(), kde=True, bins=30)
    plt.title(f"Distribution de {var}")
    plt.xlabel(var)
    plt.ylabel("Fréquence")
    plt.show()

In [ ]:
# Vous pouvez aussi visualiser les boxplots pour détecter d'éventuels outliers
for var in key_vars:
    plt.figure(figsize=(8, 2))
    sns.boxplot(x=df[var].dropna())
    plt.title(f"Boxplot de {var}")
    plt.show()

## 4. Normalisation des variables numériques

In [ ]:
# Nous utiliserons StandardScaler pour normaliser les variables clés.
scaler = StandardScaler()

# Copier le DataFrame pour y ajouter les versions normalisées
df_norm = df.copy()

for var in key_vars:
    # On crée une nouvelle colonne nommée par exemple "Production_norm"
    df_norm[var + "_norm"] = scaler.fit_transform(df[[var]])

print("\nAperçu des variables normalisées :")
print(df_norm[[var + "_norm" for var in key_vars]].head())

In [ ]:
# Visualisation : Histogrammes des variables normalisées
for var in key_vars:
    plt.figure(figsize=(8,4))
    sns.histplot(df_norm[var + "_norm"], kde=True, bins=30)
    plt.title(f"Distribution de {var} (normalisé)")
    plt.xlabel(f"{var} (normalisé)")
    plt.ylabel("Fréquence")
    plt.show()

## 5. Sélection de la période d'étude (Panel)

In [ ]:
# Garder uniquement la période de 2003-01 à 2023-12
df_panel = df_norm[(df_norm['Date'] >= '2003-01-01') & (df_norm['Date'] <= '2023-12-31')]
print("\nPériode d'étude (panel) :", df_panel['Date'].min(), "à", df_panel['Date'].max())
print("Nombre de mines :", df_panel['Mine'].nunique())

# Sauvegarder le DataFrame préparé pour la suite des analyses
df_panel.to_csv("final_merged_data_normalized.csv", index=False)


## 6. Statistiques descriptives

In [ ]:
desc_stats = df_panel[key_vars].describe()
print("\nStatistiques descriptives :")
print(desc_stats)

## 7. Corrélation

In [ ]:
# Matrice de corrélation
corr_matrix = df_panel[key_vars].corr()
print("\nMatrice de corrélation :")
print(corr_matrix)


In [ ]:
# Visualisation de la matrice de corrélation
plt.figure(figsize=(14, 12))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", square=True, cbar=True)
plt.title("Heatmap de la matrice de corrélation")
plt.show()